In [1]:
import torch

In [2]:
cd ..

e:\vshc\vhsc_multimodal_vhsc


e:\vshc\venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [7]:
type(model).__name__

'BiModalRegressor'

In [3]:
model = torch.load("data/checkpoints/toy.pth", weights_only=False)
model.requires_grad_(False)

BiModalRegressor(
  (x1_encoder): MLP(
    (net): Sequential(
      (0): Linear(in_features=1, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.5, inplace=False)
      (3): Linear(in_features=64, out_features=64, bias=True)
      (4): GELU(approximate='none')
      (5): Dropout(p=0.5, inplace=False)
      (6): Linear(in_features=64, out_features=32, bias=True)
    )
  )
  (x2_encoder): MLP(
    (net): Sequential(
      (0): Linear(in_features=1, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.5, inplace=False)
      (3): Linear(in_features=64, out_features=64, bias=True)
      (4): GELU(approximate='none')
      (5): Dropout(p=0.5, inplace=False)
      (6): Linear(in_features=64, out_features=32, bias=True)
    )
  )
  (head): MLP(
    (net): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.5, inplace=False)
      (3): Linear

In [4]:
print(model)

BiModalRegressor(
  (x1_encoder): MLP(
    (net): Sequential(
      (0): Linear(in_features=1, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.5, inplace=False)
      (3): Linear(in_features=64, out_features=64, bias=True)
      (4): GELU(approximate='none')
      (5): Dropout(p=0.5, inplace=False)
      (6): Linear(in_features=64, out_features=32, bias=True)
    )
  )
  (x2_encoder): MLP(
    (net): Sequential(
      (0): Linear(in_features=1, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.5, inplace=False)
      (3): Linear(in_features=64, out_features=64, bias=True)
      (4): GELU(approximate='none')
      (5): Dropout(p=0.5, inplace=False)
      (6): Linear(in_features=64, out_features=32, bias=True)
    )
  )
  (head): MLP(
    (net): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.5, inplace=False)
      (3): Linear

In [5]:
from src.plugins.hook import Breakpoint, BreakpointController
from src.plugins.reconstructor.linear import BilinearReconstructor

In [6]:
recon_net = BilinearReconstructor(d_1=64,
                                  d_2=64,
                                  concat=True,
                                  dropout=0.3).cuda()

0.3 relu none 32
0.3 relu none 64
0.3 relu none 32
0.3 relu none 64
0.3 relu none 64
0.3 relu none 64
0.3 relu none 64
0.3 relu none 64


In [7]:
bp = Breakpoint("recon_net", callback=recon_net, mutate=True, valid=True)

In [8]:
controller = BreakpointController()

In [9]:
controller.add_breakpoint(root=model,
                          target="head",
                          bp=bp,
                          position="before")

In [10]:
bp.kwargs = (0, 1)

In [11]:
x1, x2 = torch.randn([64]).cuda(), torch.randn([64]).cuda()

In [12]:
y = model(x1, x2)

In [13]:
print(bp.trace)

BreakpointOutput(
  fn_name='BilinearReconstructor.forward'
  output=  [0]: Tensor(shape=(64, 128), dtype=torch.float32, device=cuda:0, requires_grad=True)
  trace=
      signal:
          [0]: 0
          [1]: 1
      input:
          [0]: Tensor(shape=(64, 64), dtype=torch.float32, device=cuda:0, requires_grad=True)
          [1]: Tensor(shape=(64, 64), dtype=torch.float32, device=cuda:0, requires_grad=True)
      reconstructed:
          [0]: Tensor(shape=(64, 64), dtype=torch.float32, device=cuda:0, requires_grad=True)
          [1]: Tensor(shape=(64, 64), dtype=torch.float32, device=cuda:0, requires_grad=True)
      distance:
          [0]: Tensor(shape=(64, 64), dtype=torch.float32, device=cuda:0, requires_grad=True)
          [1]: Tensor(shape=(64, 64), dtype=torch.float32, device=cuda:0, requires_grad=False)
      dev:
          [0]: Tensor(shape=(64, 64), dtype=torch.float32, device=cuda:0, requires_grad=True)
          [1]: Tensor(shape=(64, 64), dtype=torch.float32, device=c